T10

In [6]:
import numpy as np
import scipy.stats as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import poisson, chi2

np.random.seed(42)

In [4]:
alpha = 0.05
k = 10
n = 100

nums = np.arange(k)
mi = np.array([5,8,6,12,14,18,11,6,13,7])


In [ ]:
# 1. Исходные данные
# Частоты для цифр 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
mi = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
n = np.sum(mi)

# 2. Группировка (Строго по правилу np_i >= 5)
# При среднем ~4.7, вероятности малых x низкие. 
# Объединим {0, 1, 2} в одну группу и {9, 10...} в "хвост"
m_obs = np.array([
    np.sum(mi[0:3]), # Группа 0: x in {0, 1, 2}
    mi[3],           # Группа 1: x = 3
    mi[4],           # Группа 2: x = 4
    mi[5],           # Группа 3: x = 5
    mi[6],           # Группа 4: x = 6
    mi[7],           # Группа 5: x = 7
    mi[8],           # Группа 6: x = 8
    mi[9]            # Группа 7: x >= 9 (хвост)
])
k = len(m_obs)

# 3. Функция правдоподобия для ОМПГ
def OMPG(lam):
    # Если вышли за границы (lambda > 0)
    if lam <= 0: return 1e10
    
    # Теоретические вероятности для групп
    p = np.zeros(k)
    p[0] = poisson.cdf(2, lam)              # P(X <= 2)
    for i in range(1, k-1):
        p[i] = poisson.pmf(i + 2, lam)      # P(X = 3), P(X = 4)...
    p[-1] = 1 - poisson.cdf(8, lam)         # P(X >= 9)
    
    # Чтобы log(0) не выдал ошибку
    p = np.clip(p, 1e-10, 1.0)
    
    # Возвращаем -ln L(lambda) = -sum(m_i * ln(p_i))
    return -np.sum(m_obs * np.log(p))

# 4. Численное нахождение оценки ОМПГ
# Начальное приближение — выборочное среднее
initial_guess = np.average(np.arange(10), weights=mi)
res = minimize(lambda x: OMPG(x[0]), x0=[initial_guess], method='Nelder-Mead')
lam_hat = res.x[0]

# 5. Расчет статистики Пирсона (Delta)
# Считаем финальные вероятности с найденным lam_hat
p_final = np.zeros(k)
p_final[0] = poisson.cdf(2, lam_hat)
for i in range(1, k-1):
    p_final[i] = poisson.pmf(i + 2, lam_hat)
p_final[-1] = 1 - poisson.cdf(8, lam_hat)

expected = n * p_final
delta = np.sum((m_obs - expected)**2 / expected)

# 6. Проверка гипотезы
# Число степеней свободы: k (групп) - 1 (параметр lam) - 1 = k - 2
df = k - 2
alpha = 0.05
critical_value = chi2.ppf(1 - alpha, df)
p_value = 1 - chi2.cdf(delta, df)

print(f"Оценка ОМПГ (lambda): {lam_hat:.4f}")
print(f"Статистика Пирсона (Delta): {delta:.4f}")
print(f"Критическое значение (chi2): {critical_value:.4f}")
print(f"P-value: {p_value:.4f}")

if delta < critical_value:
    print("Результат: Гипотеза H0 (Пуассон) принимается.")
else:
    print("Результат: Гипотеза H0 (Пуассон) отвергается.")

Оценка ОМПГ (lam): 4.9316
Статистика Пирсона: 13.7592
Критическое хи-квадрат (df=6): 12.5916
